# Paper-to-Project: Phase 1–8 Multi-Paper Integration Test

This notebook runs the **complete backend pipeline** (Phase 1 to Phase 8) over all research papers and saves a consolidated verification report.

| Phase | Day(s) | Description |
|-------|--------|-------------|
| Phase 1 | Day 1–4 | PDF Extraction & Multi-Engine Routing |
| Phase 2 | Day 5–8 | Canonical PaperDocument Merge & Section Audit |
| Phase 3 | Day 9–13 | Confidence, Provenance & Ingestion Benchmarks |
| Phase 4 | Day 14–18 | Semantic Chunking, Embeddings & RAG Retrieval |
| Phase 5 | Day 19–22 | Ingestion, Decomposition, Parameters, Gap Finding |
| Phase 6 | Day 23–27 | Hardware profiler, Resource estimation, Feasibility Engine, Refinement, sequencing |
| Phase 7 | Day 28–30 | Project Specification, File Planning, Component-Level Code Generation |
| Phase 8 | Day 31–33 | AST Static checks, PyTorch Automated shape tests, Paper-to-Code verification |

## Cell 0: Environment Setup — Discover All PDFs

In [ ]:
import os
import sys
import json
import time
import datetime
import traceback
from collections import Counter

# ---- PATH SETUP ----
NOTEBOOK_DIR = os.path.abspath('')
if os.path.basename(NOTEBOOK_DIR) == 'tests':
    BACKEND_DIR = os.path.dirname(NOTEBOOK_DIR)
else:
    BACKEND_DIR = NOTEBOOK_DIR

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

PAPERS_DIR  = os.path.join(BACKEND_DIR, 'papers', 'research_papers')
REPORTS_DIR = os.path.join(BACKEND_DIR, 'tests', 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# ---- DISCOVER ALL PDFs ----
all_pdfs = sorted(
    [os.path.join(PAPERS_DIR, f) for f in os.listdir(PAPERS_DIR) if f.lower().endswith('.pdf')],
    key=lambda p: int(os.path.basename(p).replace('[', '').replace('].pdf', ''))
)

print(f'[ENV] Backend dir : {BACKEND_DIR}')
print(f'[ENV] Papers dir  : {PAPERS_DIR}')
print(f'[ENV] Reports dir : {REPORTS_DIR}')
print(f'[ENV] PDFs found  : {len(all_pdfs)}')
print()
for i, p in enumerate(all_pdfs, 1):
    print(f'  [{i:>2}] {os.path.basename(p)}')

## Cell 1: Auto-Detect System Configuration

All values are **auto-detected** from your machine:
- **Model** → queried from the local Ollama REST API
- **GPU / VRAM** → detected via `torch.cuda` then `nvidia-smi` fallback
- **System RAM** → detected via `psutil`
- **Timeline** → only value you need to set manually

In [ ]:
import subprocess
import requests

# ================================================================
# AUTO-DETECT: Ollama Model
# Queries the local Ollama REST API and picks the best available.
# ================================================================
PREFERRED_MODELS = [
    'qwen2.5-coder:1.5b', 'qwen2.5-coder:7b',
    'llama3', 'mistral', 'gemma'
]
MODEL_NAME = None
try:
    resp = requests.get('http://localhost:11434/api/tags', timeout=5)
    available_models = [m['name'] for m in resp.json().get('models', [])]
    for pref in PREFERRED_MODELS:
        if pref in available_models:
            MODEL_NAME = pref
            break
    if not MODEL_NAME and available_models:
        MODEL_NAME = available_models[0]
    print(f'[AUTO] Ollama models available : {available_models}')
    print(f'[AUTO] Selected model          : {MODEL_NAME}')
except Exception as e:
    MODEL_NAME = 'qwen2.5-coder:1.5b'
    print(f'[WARN] Ollama API unreachable ({e}). Fallback: {MODEL_NAME}')

# ================================================================
# AUTO-DETECT: System RAM via psutil
# ================================================================
try:
    import psutil
    system_ram_gb = round(psutil.virtual_memory().total / (1024 ** 3), 1)
    print(f'[AUTO] System RAM              : {system_ram_gb} GB')
except ImportError:
    system_ram_gb = 16.0
    print(f'[WARN] psutil not installed. Defaulting RAM = {system_ram_gb} GB')

# ================================================================
# AUTO-DETECT: GPU Name + VRAM
# Tries torch.cuda first, then nvidia-smi as fallback.
# ================================================================
gpu_model = 'CPU (no GPU detected)'
vram_gb   = 0.0
try:
    import torch
    if torch.cuda.is_available():
        gpu_model = torch.cuda.get_device_name(0)
        vram_gb   = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1)
        print(f'[AUTO] GPU  (torch)            : {gpu_model}')
        print(f'[AUTO] VRAM (torch)            : {vram_gb} GB')
    else:
        raise RuntimeError('CUDA not available in torch')
except Exception as torch_err:
    print(f'[WARN] torch CUDA failed ({torch_err}). Trying nvidia-smi...')
    try:
        smi_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            timeout=5, text=True
        ).strip().splitlines()[0]
        smi_mem = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            timeout=5, text=True
        ).strip().splitlines()[0]
        gpu_model = smi_name
        vram_gb   = round(int(smi_mem) / 1024, 1)   # MiB -> GiB
        print(f'[AUTO] GPU  (nvidia-smi)       : {gpu_model}')
        print(f'[AUTO] VRAM (nvidia-smi)       : {vram_gb} GB')
    except Exception as smi_err:
        print(f'[WARN] nvidia-smi failed ({smi_err}). GPU constraints = 0.')

# ================================================================
# MANUAL: Only timeline needs human input
# ================================================================
TIMELINE_WEEKS = 2   # <-- change this if needed

CONSTRAINTS = {
    'gpu_model'      : gpu_model,
    'system_ram_gb'  : system_ram_gb,
    'vram_gb'        : vram_gb,
    'timeline_weeks' : TIMELINE_WEEKS
}

# ================================================================
# SCOPE: Set PAPER_LIMIT to None for all 29, or an integer
# (e.g. 1) for a quick test before the full run.
# ================================================================
PAPER_LIMIT   = None   # None = all papers
SKIP_ON_ERROR = True   # True = log errors and continue

papers_to_run = all_pdfs[:PAPER_LIMIT] if PAPER_LIMIT else all_pdfs

print()
print('--- Final Configuration ---')
print(f'  Model          : {MODEL_NAME}')
print(f'  GPU            : {gpu_model}')
print(f'  VRAM           : {vram_gb} GB')
print(f'  System RAM     : {system_ram_gb} GB')
print(f'  Timeline       : {TIMELINE_WEEKS} weeks')
print(f'  Papers to run  : {len(papers_to_run)}')
print(f'  Skip on error  : {SKIP_ON_ERROR}')

## Cell 2: Import Pipeline Orchestrator

In [ ]:
from pipeline import graph as orchestrator
print('[OK] Pipeline orchestrator imported successfully.')

## Cell 3: Run Pipeline Over All Papers (Including Phase 8 Verification)

In [ ]:
all_results = []
all_errors = []

run_start_time = time.time()
RUN_TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

print("=" * 65)
print(f"RUNNING COMPLETE PHASES 1-8 PIPELINE FOR {len(papers_to_run)} PAPERS")
print(f"Model: {MODEL_NAME} | GPU: {CONSTRAINTS['gpu_model']}")
print("=" * 65)

for paper_idx, pdf_path in enumerate(papers_to_run, 1):
    pdf_name = os.path.basename(pdf_path)
    paper_id = f"paper_{pdf_name.replace('[', '').replace('].pdf', '')}"

    print()
    print("=" * 65)
    print(f"[{paper_idx:>2}/{len(papers_to_run)}] Processing: {pdf_name}")
    print("=" * 65)

    initial_state = {
        'pdf_path': pdf_path,
        'constraints': CONSTRAINTS,
        'model_name': MODEL_NAME,
        'loop_count': 0
    }

    t0 = time.time()
    try:
        # Run graph (now automatically executes Ingestion up to Phase 8 Day 33 checks)
        result = orchestrator.invoke(initial_state)
        elapsed = round(time.time() - t0, 2)

        meta = result.get('metadata')
        pdoc = result.get('paper_doc')
        cg = result.get('component_graph')
        ext_params = result.get('extracted_parameters')
        gap_rpt = result.get('gap_report')
        res_est = result.get('resource_estimation')
        feat = result.get('feasibility_report')
        bseq = result.get('build_sequence')
        spec = result.get('project_specification')
        tree = result.get('project_tree')
        sc_rpt = result.get('static_check_report')
        tc_rpt = result.get('automated_test_report')
        vc_rpt = result.get('code_verification_report')
        arpt = result.get('report')

        gap_counts = Counter(g.classification for g in gap_rpt.parameter_gaps) if gap_rpt else {}
        param_status_counts = dict(Counter(
            getattr(ext_params, f).status for f in ext_params.__class__.model_fields.keys()
        )) if ext_params else {}

        # Capture adaptations
        adaptations = []
        if cg:
            for comp in cg.components:
                for param_name, param_details in comp.parameters.items():
                    if "PAPER ORIGINAL" in param_details.rationale:
                        adaptations.append({
                            'component': comp.name,
                            'parameter': param_name,
                            'value': param_details.value,
                            'trace': param_details.rationale
                        })

        paper_result = {
            'paper_id': paper_id,
            'pdf_name': pdf_name,
            'status': 'SUCCESS',
            'elapsed_seconds': elapsed,
            'title': meta.title if meta else 'Unknown Remote Sensing Research Paper',
            'authors': meta.authors if meta else [],
            'abstract': meta.abstract if meta else '',
            'primary_contribution': meta.primary_contribution if meta else '',
            'sections_count': len(pdoc.sections) if pdoc else 0,
            'tables_count': len(pdoc.tables) if pdoc else 0,
            'equations_count': len(pdoc.equations) if pdoc else 0,
            'components_count': len(cg.components) if cg else 0,
            'edges_count': len(cg.edges) if cg else 0,
            'param_status_counts': param_status_counts,
            'gap_counts': dict(gap_counts),
            'feasibility_status': feat.overall_status if feat else 'UNKNOWN',
            'milestones_count': len(bseq.milestones) if bseq else 0,
            'total_duration_weeks': getattr(bseq, 'total_duration_weeks', 0.0) if bseq else 0.0,
            'adaptations': adaptations,
            'generated_files_count': len(tree.files) if tree else 0,
            # Phase 8 validation metrics
            'static_checks': {
                'syntax': sc_rpt.syntax_valid if sc_rpt else False,
                'imports': sc_rpt.imports_valid if sc_rpt else False,
                'dependencies': sc_rpt.dependencies_valid if sc_rpt else False
            },
            'automated_tests': {
                'dataset': tc_rpt.dataset_check if tc_rpt else False,
                'backbone': tc_rpt.backbone_check if tc_rpt else False,
                'fusion': tc_rpt.fusion_check if tc_rpt else False,
                'decoder': tc_rpt.decoder_check if tc_rpt else False,
                'loss': tc_rpt.loss_check if tc_rpt else False
            },
            'verification_traces': vc_rpt.comparisons if vc_rpt else [],
            '_result_full': result
        }

        all_results.append(paper_result)
        print(f"  [OK] Done in {elapsed}s")

    except Exception as e:
        elapsed = round(time.time() - t0, 2)
        err_msg = str(e)
        print(f"  [ERROR] {pdf_name} failed after {elapsed}s: {err_msg}")
        err_data = {
            'paper_id': paper_id,
            'pdf_name': pdf_name,
            'status': 'ERROR',
            'elapsed_seconds': elapsed,
            'error': err_msg,
            'traceback': traceback.format_exc()
        }
        all_errors.append(err_data)

total_run_time = round(time.time() - run_start_time, 2)
print()
print('=' * 65)
print(f'RUN COMPLETE: {len(all_results)} success, {len(all_errors)} errors')
print(f'Total time  : {total_run_time}s ({round(total_run_time/60, 1)} min)')
print('=' * 65)

## Cell 4: Per-Paper Summary Table (Phases 1-8)

In [ ]:
header = f"{'#':<3} {'PDF':<10} {'STATUS':<7} {'TIME(s)':<8} {'FEASIBILITY':<13} {'STATIC (S/I/D)':<16} {'AUTOMATED (D/B/F/C/L)':<23} {'ADAPT.':<6} {'FILES':<5}"
print(header)
print('-' * 105)
for i, r in enumerate(all_results, 1):
    sc = r['static_checks']
    tc = r['automated_tests']
    
    sc_str = f"{'✓' if sc['syntax'] else '✗'}/{'✓' if sc['imports'] else '✗'}/{'✓' if sc['dependencies'] else '✗'}"
    tc_str = f"{'✓' if tc['dataset'] else '✗'}/{'✓' if tc['backbone'] else '✗'}/{'✓' if tc['fusion'] else '✗'}/{'✓' if tc['decoder'] else '✗'}/{'✓' if tc['loss'] else '✗'}"
    
    print(
        f"{i:<3} {r['pdf_name']:<10} {'OK':<7} {r['elapsed_seconds']:<8.1f} "
        f"{r['feasibility_status']:<13} {sc_str:<16} {tc_str:<23} "
        f"{len(r['adaptations']):<6} {r['generated_files_count']:<5}"
    )
for r in all_errors:
    print(
        f"{'--':<3} {r['pdf_name']:<10} {'ERROR':<7} {r['elapsed_seconds']:<8.1f} "
        f"{'N/A':<13} {'--':<16} {'--':<23} {'--':<6} {'--':<5}"
    )
print()
print(f'Total: {len(all_results)} success / {len(all_errors)} errors / {len(papers_to_run)} total')

## Cell 5: Corpus-Wide Aggregate Statistics

In [ ]:
if all_results:
    all_feasibility  = Counter(r['feasibility_status']  for r in all_results)
    all_gap_counts   = Counter()
    all_param_status = Counter()
    total_components = sum(r['components_count'] for r in all_results)
    total_edges      = sum(r['edges_count'] for r in all_results)
    avg_elapsed      = round(sum(r['elapsed_seconds'] for r in all_results) / len(all_results), 1)
    
    # Verification stats
    total_syntax_pass = sum(1 for r in all_results if r['static_checks']['syntax'])
    total_imports_pass = sum(1 for r in all_results if r['static_checks']['imports'])
    total_deps_pass = sum(1 for r in all_results if r['static_checks']['dependencies'])
    total_test_pass = sum(1 for r in all_results if all(r['automated_tests'].values()))

    for r in all_results:
        all_gap_counts   += Counter(r.get('gap_counts', {}))
        all_param_status += Counter(r.get('param_status_counts', {}))
    critical_missing_count = sum(1 for r in all_results if r.get('gap_counts', {}).get('MISSING', 0) > 0)

    print('=' * 55)
    print('CORPUS-WIDE AGGREGATE STATISTICS')
    print('=' * 55)
    print(f'  System            : {gpu_model}')
    print(f'  VRAM              : {vram_gb} GB | RAM: {system_ram_gb} GB')
    print(f'  Model             : {MODEL_NAME}')
    print()
    print(f'  Papers processed  : {len(all_results)} / {len(papers_to_run)}')
    print(f'  Papers failed     : {len(all_errors)}')
    print(f'  Avg time / paper  : {avg_elapsed}s')
    print()
    print('  --- Verification Check Success Rates ---')
    print(f'    AST Syntax Checks: {total_syntax_pass}/{len(all_results)} ({round(total_syntax_pass/len(all_results)*100)}%)')
    print(f'    Imports Checks   : {total_imports_pass}/{len(all_results)} ({round(total_imports_pass/len(all_results)*100)}%)')
    print(f'    Dependency Map   : {total_deps_pass}/{len(all_results)} ({round(total_deps_pass/len(all_results)*100)}%)')
    print(f'    Dynamic Model IO : {total_test_pass}/{len(all_results)} ({round(total_test_pass/len(all_results)*100)}%)')
    print()
    print('  --- Feasibility Distribution ---')
    for k, v in sorted(all_feasibility.items()):
        print(f'    {k:<15}: {v} paper(s)')
    print()
    print('  --- Gap Classification Distribution ---')
    for k, v in sorted(all_gap_counts.items()):
        print(f'    {k:<15}: {v} parameters')
    print(f'  Critical missing in {critical_missing_count} paper(s)')

## Cell 6: Save Consolidated Report (Markdown + JSON)

In [ ]:
import os
import datetime
import json

# Setup output paths for Phase 1 to 8
report_md_path = os.path.join(REPORTS_DIR, f'Phase_1_to_8.md')
report_json_path = os.path.join(REPORTS_DIR, f'Phase_1_to_8.json')

total_run_time = round(time.time() - run_start_time, 2) if 'run_start_time' in globals() else 0.0
avg_elapsed = round(sum(r['elapsed_seconds'] for r in all_results) / len(all_results), 1) if all_results else 0
total_components = sum(r['components_count'] for r in all_results)
total_edges = sum(r['edges_count'] for r in all_results)
all_feasibility = Counter(r['feasibility_status'] for r in all_results)

all_gap_counts = Counter()
all_resource_tiers = Counter()
critical_missing_count = 0
total_files_generated = 0

for r in all_results:
    all_gap_counts += Counter(r.get('gap_counts', {}))
    total_files_generated += r.get('generated_files_count', 0)
    res_est = r['_result_full'].get('resource_estimation') if '_result_full' in r else None
    if res_est:
        all_resource_tiers[res_est.overall_resource_tier] += 1
    if r.get('gap_counts', {}).get('MISSING', 0) > 0:
        critical_missing_count += 1

all_param_status = Counter()
for r in all_results:
    all_param_status += Counter(r.get('param_status_counts', {}))

# ==============================================================
#  MARKDOWN REPORT GENERATION
# ==============================================================
md = []
md.append('# Paper-to-Project: Phase 1–8 Corpus-Wide Code Synthesis & Verification Report')
md.append(f'**Generated:** {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
md.append(f'**Model:** `{MODEL_NAME}`' if 'MODEL_NAME' in globals() else '**Model:** `qwen2.5-coder:1.5b`')
md.append(f'**GPU:** {gpu_model} ({vram_gb} GB VRAM) | **RAM:** {system_ram_gb} GB' if 'gpu_model' in globals() else '')
md.append(f'**Total Running Time:** {total_run_time} seconds')
md.append(f'**Papers Run:** {len(papers_to_run)} | **Success:** {len(all_results)} | **Errors:** {len(all_errors)}' if 'papers_to_run' in globals() else '')
md.append('')

# Summary table
md.append('## Per-Paper Verification Scorecard')
md.append('| # | PDF | Feasibility | Static AST (S/I/D) | Automated Tests (D/B/F/C/L) | Code Files | Verification Status |')
md.append('|---|-----|-------------|--------------------|----------------------------|------------|---------------------|')
for i, r in enumerate(all_results, 1):
    sc = r['static_checks']
    tc = r['automated_tests']
    sc_str = f"{'✓' if sc['syntax'] else '✗'}/{'✓' if sc['imports'] else '✗'}/{'✓' if sc['dependencies'] else '✗'}"
    tc_str = f"{'✓' if tc['dataset'] else '✗'}/{'✓' if tc['backbone'] else '✗'}/{'✓' if tc['fusion'] else '✗'}/{'✓' if tc['decoder'] else '✗'}/{'✓' if tc['loss'] else '✗'}"
    overall_ver = "✓ VERIFIED" if (all(sc.values()) and all(tc.values())) else "⚠ WARNINGS"
    md.append(
        f"| {i} | {r['pdf_name']} | {r['feasibility_status']} | "
        f"{sc_str} | {tc_str} | {r['generated_files_count']} | {overall_ver} |"
    )
md.append('')

# Per-paper details section
md.append('## Per-Paper Code Verification Traces')
md.append('')
for i, r in enumerate(all_results, 1):
    res = r['_result_full']
    spec = res.get('project_specification')
    tree = res.get('project_tree')
    
    md.append(f'### [{i}] {r["pdf_name"]} — {r["title"][:80]}')
    md.append('')
    md.append(f'- **Time:** {r["elapsed_seconds"]}s | **Feasibility:** {r["feasibility_status"]}')
    
    if spec:
        md.append(f'- **Architecture:** {spec.architecture}')
        md.append(f'- **System Requirements:** {spec.requirements}')
        
    if r['verification_traces']:
        md.append('\n**Paper ↔ Code Verification Traces:**')
        for trace in r['verification_traces']:
            md.append(f'- {trace}')
            
    if tree:
        md.append('\n**ASCII Project Structure Layout:**')
        md.append('```text')
        md.append(tree.tree_structure)
        md.append('```')
    md.append('---')

with open(report_md_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(md))
print(f'[OK] Markdown report : {report_md_path}')


# ==============================================================
#  JSON REPORT SERIALIZATION
# ==============================================================
def safe_serialize(r):
    res  = r.get('_result_full', {})
    cg   = res.get('component_graph')
    grpt = res.get('gap_report')
    res_est = res.get('resource_estimation')
    feat = res.get('feasibility_report')
    bseq = res.get('build_sequence')
    extp = res.get('extracted_parameters')
    spec = res.get('project_specification')
    tree = res.get('project_tree')
    pdoc = res.get('paper_doc')
    arpt = res.get('report')
    meta = res.get('metadata')
    
    components_list = []
    if feat:
        cf_list = getattr(feat, 'components_analysis', getattr(feat, 'components', []))
        if cf_list:
            for cf in cf_list:
                components_list.append({
                    'name': getattr(cf, 'component_name', getattr(cf, 'name', 'N/A')) if not isinstance(cf, dict) else cf.get('component_name', cf.get('name', 'N/A')),
                    'status': getattr(cf, 'status', 'N/A') if not isinstance(cf, dict) else cf.get('status', 'N/A'),
                    'reason': getattr(cf, 'reason', 'N/A') if not isinstance(cf, dict) else cf.get('reason', 'N/A')
                })

    return {
        'paper_id'        : r['paper_id'],
        'pdf_name'        : r['pdf_name'],
        'status'          : r['status'],
        'elapsed_seconds' : r['elapsed_seconds'],
        'metadata': {
            'title'   : meta.title    if meta else 'N/A',
            'authors' : meta.authors  if meta else [],
            'abstract': meta.abstract[:500] if meta else '',
            'primary_contribution': meta.primary_contribution if meta else ''
        },
        'paper_doc_stats': {
            'sections' : len(pdoc.sections)  if pdoc else 0,
            'tables'   : len(pdoc.tables)    if pdoc else 0,
            'equations': len(pdoc.equations) if pdoc else 0
        },
        'component_graph': {
            'components': [{'name': c.name, 'type': c.type, 'params': list(c.parameters.keys())} for c in cg.components] if cg else [],
            'edges'     : cg.edges if cg else []
        },
        'extracted_parameters': {
            field: {'value': getattr(extp, field).value, 'status': getattr(extp, field).status, 'confidence': getattr(extp, field).confidence}
            for field in extp.__class__.model_fields.keys()
        } if extp else {},
        'gap_report': {
            'summary'             : grpt.summary if grpt else '',
            'has_critical_missing': grpt.has_critical_missing_parameters if grpt else None,
            'gaps': [{'parameter': g.parameter_name, 'classification': g.classification, 'value': g.value, 'details': g.details} for g in grpt.parameter_gaps] if grpt else []
        },
        'resource_estimation': {
            'param_count_millions': res_est.model.param_count_millions if res_est else 0.0,
            'weights_mb': res_est.model.model_weights_mb if res_est else 0.0,
            'vram_recommended_gb': res_est.training.vram_recommended_gb if res_est else 0.0,
            'overall_resource_tier': res_est.overall_resource_tier if res_est else 'UNKNOWN'
        } if res_est else {},
        'feasibility': {
            'overall_status'     : feat.overall_status      if feat else 'N/A',
            'training_status'    : feat.training_status     if feat else 'N/A',
            'training_substitute': feat.training_substitute if feat else '',
            'components': components_list
        },
        'project_specification': {
            'requirements': spec.requirements if spec else '',
            'architecture': spec.architecture if spec else '',
            'components': spec.components if spec else []
        } if spec else {},
        'project_tree': {
            'files_count': len(tree.files) if tree else 0,
            'structure': tree.tree_structure if tree else ''
        } if tree else {},
        'verification_reports': {
            'static_checks': r['static_checks'],
            'automated_tests': r['automated_tests'],
            'paper_code_verification': r['verification_traces']
        }
    }

json_report = {
    'generated_at'    : datetime.datetime.now().isoformat(),
    'system': {
        'model'         : MODEL_NAME if 'MODEL_NAME' in globals() else 'qwen2.5-coder:1.5b',
        'gpu'           : gpu_model if 'gpu_model' in globals() else 'CPU Only',
        'vram_gb'       : vram_gb if 'vram_gb' in globals() else 0.0,
        'system_ram_gb' : system_ram_gb if 'system_ram_gb' in globals() else 16.0,
        'timeline_weeks': TIMELINE_WEEKS if 'TIMELINE_WEEKS' in globals() else 2
    },
    'papers_total'    : len(papers_to_run) if 'papers_to_run' in globals() else len(all_results),
    'papers_success'  : len(all_results),
    'papers_error'    : len(all_errors),
    'total_runtime_sec': total_run_time,
    'aggregate': {
        'total_components'     : total_components if all_results else 0,
        'total_edges'          : total_edges      if all_results else 0,
        'avg_elapsed_seconds'  : avg_elapsed      if all_results else 0,
        'feasibility_dist'     : dict(all_feasibility)  if all_results else {},
        'gap_class_dist'       : dict(all_gap_counts)   if all_results else {},
        'param_status_dist'    : dict(all_param_status) if all_results else {},
        'resource_tier_dist'   : dict(all_resource_tiers) if all_results else {},
        'total_files_generated': total_files_generated,
        'critical_missing_count': critical_missing_count if all_results else 0,
        'static_checks_success': {
            'syntax_rate': f"{total_syntax_pass}/{len(all_results)}",
            'imports_rate': f"{total_imports_pass}/{len(all_results)}",
            'dependencies_rate': f"{total_deps_pass}/{len(all_results)}"
        },
        'automated_tests_rate': f"{total_test_pass}/{len(all_results)}"
    },
    'papers': [safe_serialize(r) for r in all_results],
    'errors': all_errors
}

with open(report_json_path, 'w', encoding='utf-8') as f:
    json.dump(json_report, f, indent=2, ensure_ascii=False)
print(f'[OK] JSON report     : {report_json_path}')

print()
print('============================')
print('ALL PHASE 1-8 REPORTS SAVED')
print('============================')


## Cell 7: Final Scorecard

In [ ]:
print('=' * 65)
print('PHASE 1-8 CORPUS SCORECARD')
print('=' * 65)
print(f"  GPU              : {gpu_model}")
print(f"  VRAM             : {vram_gb} GB | RAM: {system_ram_gb} GB")
print(f"  Model            : {MODEL_NAME}")
print()
print(f"  Papers total     : {len(papers_to_run)}")
print(f"  Success          : {len(all_results)}")
print(f"  Errors           : {len(all_errors)}")
if all_errors:
    print(f"  Failed           : {', '.join(e['pdf_name'] for e in all_errors)}")
print()
if all_results:
    print(f"  Avg time/paper   : {avg_elapsed}s")
    print(f"  Total components : {total_components}")
    print(f"  Total Py Files   : {total_files_generated} files generated")
    print()
    print(f"  --- Verification Success Rates ---")
    print(f"    AST Syntax Checks : {total_syntax_pass}/{len(all_results)} ({round(total_syntax_pass/len(all_results)*100)}%)")
    print(f"    Imports Resolution: {total_imports_pass}/{len(all_results)} ({round(total_imports_pass/len(all_results)*100)}%)")
    print(f"    Dynamic IO Tests  : {total_test_pass}/{len(all_results)} ({round(total_test_pass/len(all_results)*100)}%)")
    print()
    print(f"  --- Feasibility Status Dist ---")
    for k, v in sorted(all_feasibility.items()):
        pct = round(v / len(all_results) * 100)
        print(f"    {k:<27}: {v:>3} ({pct}%)")
print()
print(f"  Reports saved to : {REPORTS_DIR}")
print(f"    MD   : Phase_1_to_8.md")
print(f"    JSON : Phase_1_to_8.json")
print('=' * 65)
print('DONE')
print('=' * 65)
